# Kaggriculture — baseline agent, local test harness, and submission

A working end-to-end setup: a rule-based agent, a local evaluation loop you can
run offline, a config sweep helper, and the packaging step.

**Order of work**
1. Install the environment and read its bundled rules
2. Write `main.py`
3. Play games locally against the built-in baselines
4. Watch a replay
5. Tune the config
6. Package and submit

The agent here scores roughly **26k–31k coins** against the built-in `starter`
baseline (~3.4k) over a full 720-turn season. That is a floor to beat, not a
finished solution — see the last cell for where the headroom is.

## 1. Setup

In [ ]:
!pip install -q -U kaggle-environments

from kaggle_environments import make
import kaggle_environments, os
print("kaggle-environments", kaggle_environments.version)

The environment ships its own rulebook. `AGENTS.md` is the API reference
(observation fields, action format); `README.md` has the crop/animal tables, the
price function, and the exact turn-processing order. Read both — nearly every
tuning decision below traces back to a number in them.

In [ ]:
ENV_DIR = os.path.join(os.path.dirname(kaggle_environments.__file__), "envs", "kaggriculture")
print(sorted(os.listdir(ENV_DIR)))

# Print the sections you need most often.
readme = open(os.path.join(ENV_DIR, "README.md")).read()
start = readme.index("## Object Types")
print(readme[start:start + 2200])

In [ ]:
# The full agent-facing API reference.
print(open(os.path.join(ENV_DIR, "AGENTS.md")).read()[:6000])

## 2. The agent

### Why it is shaped this way

**Labour is almost free.** A farm hand costs `fib(n)` coins where `n` is the
number already hired that day: 1, 1, 2, 3, 5, 8, 13, 21. A crew of ten costs
about 143 coins for the day and gives you ~240 extra actions. Against a 3,000
starting bank this is the single biggest lever in the game, and the built-in
`starter` agent ignores it entirely (it farms one tile with one farmer and never
moves). Hiring resets daily, so you re-hire every morning.

**Tiles are the real constraint.** Each unit needs somewhere useful to be, so we
buy the NE / SW / SE quadrants ($1k / $2k / $4k) as early as the cash reserve
allows.

**Watering is non-negotiable.** Two consecutive unwatered days turn a plant into
a weed, and the planting day itself counts as the first missed day. So `WATER`
outranks everything except nothing — a lost plant costs the seed plus the whole
occupancy.

**Selling needs throttling.** Prices move against you as market inventory grows,
and the curve shape differs per product. Wheat uses a log glut curve
(`above_target` 0.20), so it absorbs enormous volume; melon uses a squared curve
at 3.60, so a few hundred units drive it to the $1 floor. Hence per-day sell
caps that are generous for staples and tight for premium goods.

**Assignment is greedy.** Each turn we enumerate every worthwhile tile action,
score it as `priority - manhattan_distance`, and hand out jobs to units in
descending order. Crude, but it naturally clusters units near their work.

In [ ]:
%%writefile main.py
"""Kaggriculture baseline agent: 'Foreman'.

Strategy in one paragraph
-------------------------
Labour is the cheapest resource in this game. A farm hand costs fib(n) coins
(1, 1, 2, 3, 5, 8, 13, 21 ...) and gives you 24 extra actions that day, so a
crew of 10 costs ~143/day and roughly 11x's your throughput. The binding
constraint is therefore tiles, not money, so we buy land early, keep every
unlocked tile planted, and treat watering as sacred (two missed days turns a
plant into a weed). Sales are throttled per crop because premium goods
(melon, strawberry, wool, milk) crash to the $1 floor on modest gluts, while
wheat absorbs almost unlimited volume.

Everything tunable lives in CONFIG.
"""

# --- Game constants (mirrored from the environment; do not edit) -----------
CROPS = {
    "WHEAT":      {"seed": 10, "first_yield_day": 2,  "max_yield_day": 4,  "ongoing": False},
    "CARROT":     {"seed": 20, "first_yield_day": 2,  "max_yield_day": 3,  "ongoing": False},
    "TOMATO":     {"seed": 50, "first_yield_day": 8,  "max_yield_day": 8,  "ongoing": True},
    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "ongoing": True},
    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "ongoing": False},
}
LAND_PRICES = [1000, 2000, 4000]
SHED_CAPACITY = 100

# --- Tunables --------------------------------------------------------------
CONFIG = {
    # Crew size. Cost of n hands/day = sum of first n Fibonacci numbers.
    "max_hands": 10,
    "hire_cash_floor": 400,      # don't hire below this bank

    # Target share of planted tiles per crop. Renormalised over whatever is
    # still plantable given the days remaining.
    "crop_mix": {"CARROT": 0.40, "WHEAT": 0.25, "MELON": 0.25, "TOMATO": 0.10},

    # Max units of each product sold per day. Premium goods are drip-fed
    # because their glut curves are quadratic.
    "sell_caps": {"WHEAT": 999, "CARROT": 999, "TOMATO": 40,
                  "STRAWBERRY": 10, "MELON": 12, "EGG": 999,
                  "MILK": 10, "WOOL": 8, "FERTILIZER": 999},
    # Never sell below this fraction of the product's opening price, unless
    # the shed is overflowing or the season is ending.
    "min_price_frac": 0.35,

    "land_reserve": 300,         # keep this much cash after buying land
    "seed_buffer": 6,            # keep this many spare seeds per active crop
    "endgame_days": 1,           # last N days: liquidate everything
}

TASK_PRIORITY = {"WATER": 100, "HARVEST": 90, "FEED": 85, "PLANT": 60, "DIG": 25}


def _fib_hire_cost(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def _quadrant_of(x, y, size):
    half = size // 2
    return ("NW" if x < half else "NE") if y < half else ("SW" if x < half else "SE")


class Foreman:
    def __init__(self):
        self.day = -1
        self.sold_today = {}
        self.opening_prices = None

    # -- helpers ------------------------------------------------------------
    def _plantable(self, days_left):
        """Crops that can still reach a first harvest before the season ends."""
        return [c for c, s in CROPS.items() if days_left >= s["first_yield_day"] + 1]

    def _pick_crop(self, tiles, size, days_left, seeds, money):
        planted = {}
        for row in tiles:
            for t in row:
                if isinstance(t, dict) and t.get("kind") == "PLANT":
                    planted[t["crop"]] = planted.get(t["crop"], 0) + 1
        total = sum(planted.values()) or 1

        candidates = [c for c in self._plantable(days_left)
                      if seeds.get(c, 0) > 0 and c in CONFIG["crop_mix"]]
        if not candidates:
            return None
        mix = {c: CONFIG["crop_mix"][c] for c in candidates}
        norm = sum(mix.values()) or 1.0
        # Largest shortfall against target share wins.
        return max(candidates, key=lambda c: mix[c] / norm - planted.get(c, 0) / total)

    def _tasks(self, farm, size, step, day, days_left, seeds):
        """Every action worth taking on the board right now."""
        tasks = []
        unlocked = set(farm["unlocked_quadrants"])
        for y, row in enumerate(farm["tiles"]):
            for x, tile in enumerate(row):
                if tile == "LOCKED" or _quadrant_of(x, y, size) not in unlocked:
                    continue
                if tile is None:
                    if seeds:
                        tasks.append((x, y, "PLANT", None))
                    continue
                kind = tile.get("kind")
                if kind == "WEED":
                    tasks.append((x, y, "DIG", None))
                elif kind == "PLANT":
                    crop = tile["crop"]
                    spec = CROPS.get(crop, {})
                    age = day - tile["planted_day"]
                    ready = False
                    if spec.get("ongoing"):
                        ready = tile.get("yield_units", 0) > 0
                    else:
                        # Take the full-yield harvest, unless the season or the
                        # plant's lifespan is about to run out.
                        dying = tile.get("max_lifespan_step") is not None and \
                            step >= tile["max_lifespan_step"] - 24
                        ripe = age >= spec.get("max_yield_day", 99)
                        early = age >= spec.get("first_yield_day", 99) and \
                            (days_left <= 1 or dying)
                        ready = (ripe or early) and tile.get("yield_units", 0) > 0
                    if ready:
                        tasks.append((x, y, "HARVEST", None))
                    elif not tile.get("watered_today"):
                        tasks.append((x, y, "WATER", None))
                elif kind in ("COOP", "PASTURE"):
                    if "animal" in tile:
                        if tile.get("yield_units", 0) > 0:
                            tasks.append((x, y, "HARVEST", None))
                        elif not tile.get("watered_today", True):
                            pass
                    # Empty structures are left alone by this baseline.
        return tasks

    # -- main ---------------------------------------------------------------
    def __call__(self, obs):
        player = obs["player"]
        farm = obs["farms"][player]
        private = obs.get("private", {}) or {}
        seeds = dict(private.get("seeds", {}))
        shed = dict(private.get("shed", {}))
        inventories = private.get("inventories", []) or [{}]
        prices = (obs.get("market", {}) or {}).get("prices", {})
        size = len(farm["tiles"])
        step, day, hour = obs.get("step", 0), obs.get("day", 0), obs.get("hour", 0)
        total_days = 30
        days_left = total_days - day
        endgame = days_left <= CONFIG["endgame_days"]

        if day != self.day:
            self.day = day
            self.sold_today = {}
        if self.opening_prices is None:
            self.opening_prices = dict(prices)

        money = farm["money"]
        market = []

        # 1. Hire the crew at first light. Each HIRE is one market order and
        #    the per-turn order cap is 10, so we spread it over three turns.
        if hour in (0, 1, 2) and money > CONFIG["hire_cash_floor"]:
            already = farm.get("hires_today", 0)
            budget = money - CONFIG["hire_cash_floor"]
            slots = 0
            while (already + slots < CONFIG["max_hands"] and slots < 5
                   and _fib_hire_cost(already + slots) <= budget):
                budget -= _fib_hire_cost(already + slots)
                slots += 1
            market.extend([["HIRE"]] * slots)

        # 2. Land. More tiles is the only way to scale, so buy as soon as we
        #    can afford it with a working reserve.
        n_extra = len(farm["unlocked_quadrants"]) - 1
        if n_extra < len(LAND_PRICES) and days_left > 6:
            cost = LAND_PRICES[n_extra]
            if money >= cost + CONFIG["land_reserve"] and len(market) < 9:
                market.append(["BUY_LAND"])

        # 3. Sell, respecting per-day caps and price floors.
        shed_total = sum(shed.values())
        overflowing = shed_total > SHED_CAPACITY * 0.8
        for item, qty in sorted(shed.items(), key=lambda kv: -prices.get(kv[0], 0)):
            if qty <= 0 or len(market) >= 10:
                continue
            price = prices.get(item, 0)
            floor = CONFIG["min_price_frac"] * self.opening_prices.get(item, price or 1)
            if price < floor and not (overflowing or endgame):
                continue
            cap = 999 if (endgame or overflowing) else CONFIG["sell_caps"].get(item, 999)
            room = cap - self.sold_today.get(item, 0)
            n = min(qty, max(0, room))
            if n > 0:
                market.append(["SELL", item, n])
                self.sold_today[item] = self.sold_today.get(item, 0) + n

        # 4. Restock seeds for whatever we still have time to grow.
        empty = sum(1 for row in farm["tiles"] for t in row if t is None)
        if empty and not endgame and len(market) < 10:
            wanted = [c for c in self._plantable(days_left) if c in CONFIG["crop_mix"]]
            wanted.sort(key=lambda c: -CONFIG["crop_mix"][c])
            spend = money * 0.5
            for crop in wanted:
                if len(market) >= 10:
                    break
                need = int(empty * CONFIG["crop_mix"][crop]) + CONFIG["seed_buffer"]
                have = seeds.get(crop, 0)
                n = min(need - have, int(spend // CROPS[crop]["seed"]))
                if n > 0:
                    market.append(["BUY_SEED", crop, n])
                    spend -= n * CROPS[crop]["seed"]
                    seeds[crop] = have + n

        # 5. Assign every unit to the best unclaimed task.
        units = [tuple(farm["farmer"])] + [tuple(h) for h in farm.get("hands", [])]
        tasks = self._tasks(farm, size, step, day, days_left, seeds)

        scored = []
        for ui, (ux, uy) in enumerate(units):
            for ti, (tx, ty, kind, _) in enumerate(tasks):
                dist = abs(tx - ux) + abs(ty - uy)
                scored.append((TASK_PRIORITY[kind] - dist, ui, ti))
        scored.sort(reverse=True)

        assigned, taken = {}, set()
        plant_budget = dict(seeds)
        for _, ui, ti in scored:
            if ui in assigned or ti in taken:
                continue
            tx, ty, kind, _ = tasks[ti]
            crop = None
            if kind == "PLANT":
                crop = self._pick_crop(farm["tiles"], size, days_left,
                                       plant_budget, money)
                if crop is None or endgame:
                    continue
                plant_budget[crop] -= 1          # never over-commit seeds
                if plant_budget[crop] <= 0:
                    plant_budget.pop(crop, None)
            assigned[ui] = (tx, ty, kind, crop)
            taken.add(ti)

        ops = []
        half = size // 2
        shed_tiles = {(half - 1, half - 1), (half, half - 1),
                      (half - 1, half), (half, half)}
        for ui, (ux, uy) in enumerate(units):
            inv = inventories[ui] if ui < len(inventories) else {}
            job = assigned.get(ui)
            if job is None:
                # Idle: dump anything carried, otherwise head home.
                if sum(inv.values()) and (ux, uy) in shed_tiles:
                    ops.append(["DROP"])
                else:
                    ops.append(self._step_toward(ux, uy, half - 1, half - 1))
                continue
            tx, ty, kind, crop = job
            if (ux, uy) == (tx, ty):
                ops.append([kind, crop] if crop else [kind])
            else:
                # Drop off a full load if we happen to be standing at the shed.
                if sum(inv.values()) >= 6 and (ux, uy) in shed_tiles:
                    ops.append(["DROP"])
                else:
                    ops.append(self._step_toward(ux, uy, tx, ty))

        return {"farmer": ops[0] if ops else ["PASS"],
                "hands": ops[1:],
                "market": market[:10]}

    @staticmethod
    def _step_toward(x, y, tx, ty):
        if x < tx:
            return ["EAST"]
        if x > tx:
            return ["WEST"]
        if y < ty:
            return ["SOUTH"]
        if y > ty:
            return ["NORTH"]
        return ["PASS"]


_foreman = Foreman()


def agent(obs):
    try:
        return _foreman(obs)
    except Exception:
        # A crashed turn is worse than a wasted one.
        return {"farmer": ["PASS"], "hands": [], "market": []}


## 3. Local evaluation

Play full seasons against the built-in agents (`pass`, `random`, `starter`) and
against a previous version of yourself. Self-play is the more informative test
once you are comfortably beating `starter`, since the ladder pairs you against
other people's agents, not against a punching bag.

A 720-turn game takes a minute or two, so keep the seed list short while
iterating.

In [ ]:
import importlib, sys, statistics
sys.path.insert(0, ".")
import main; importlib.reload(main)


def play(a, b, seed=0, debug=False):
    env = make("kaggriculture", configuration={"seed": seed}, debug=debug)
    env.run([a, b])
    final = env.steps[-1]
    return env, final[0].reward, final[1].reward, final[0].status


def evaluate(agent_fn, opponents=("starter",), seeds=(1, 2, 3)):
    rows = []
    for opp in opponents:
        scores = []
        for s in seeds:
            _, mine, theirs, status = play(agent_fn, opp, seed=s)
            scores.append(mine)
            flag = "" if status == "DONE" else f"  <-- {status}"
            print(f"  vs {opp:<9} seed {s}: {mine:>9,.0f}  |  opp {theirs:>8,.0f}{flag}")
        rows.append((opp, statistics.mean(scores)))
    print()
    for opp, m in rows:
        print(f"mean vs {opp:<9} {m:>10,.0f}")
    return rows


evaluate(main.agent, opponents=("starter", "random"), seeds=(1, 2))

### Debugging a game

Set `debug=True` to surface exceptions and invalid actions. Invalid actions are
**silent no-ops** in this environment, which is the main way an agent quietly
wastes a whole season — a unit that keeps issuing `WATER` on a locked tile looks
busy and produces nothing. Worth checking how many of your actions actually
land.

In [ ]:
env, mine, theirs, status = play(main.agent, "starter", seed=7, debug=True)
print(status, mine, theirs)

# Money curve over the season — where does growth stall?
money = [s[0]["observation"]["farms"][0]["money"] for s in env.steps]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 3))
    plt.plot(money)
    plt.xlabel("turn"); plt.ylabel("bank"); plt.title("Bank over the season")
    plt.grid(alpha=.3); plt.show()
except ImportError:
    print(money[::48])

In [ ]:
# How many tiles are actually in production, day by day?
def tile_census(env, seat=0):
    size = len(env.steps[0][0]["observation"]["farms"][seat]["tiles"])
    for step in range(0, len(env.steps), 24 * 5):
        farm = env.steps[step][0]["observation"]["farms"][seat]
        flat = [t for row in farm["tiles"] for t in row]
        planted = sum(1 for t in flat if isinstance(t, dict) and t.get("kind") == "PLANT")
        weeds = sum(1 for t in flat if isinstance(t, dict) and t.get("kind") == "WEED")
        empty = sum(1 for t in flat if t is None)
        locked = sum(1 for t in flat if t == "LOCKED")
        print(f"day {step // 24:>2}  planted {planted:>3}  weeds {weeds:>2}  "
              f"empty {empty:>3}  locked {locked:>3}  money {farm['money']:>8,.0f}")

tile_census(env)

## 4. Watch a replay

The visualiser is the fastest way to spot a stupid loop — units pacing between
two tiles, a quadrant sitting empty, harvests rotting because nobody walks
there.

In [ ]:
env.render(mode="ipython", width=1000, height=700)

## 5. Tuning

Most of your leaderboard movement will come from these numbers, not from
rewriting the architecture. Change one thing at a time and keep the seed set
fixed so the comparison means something.

In [ ]:
import itertools, copy

BASE = copy.deepcopy(main.CONFIG)


def sweep(key, values, seeds=(1, 2), opponent="starter"):
    results = {}
    for v in values:
        importlib.reload(main)
        main.CONFIG.update(BASE)
        main.CONFIG[key] = v
        scores = [play(main.agent, opponent, seed=s)[1] for s in seeds]
        results[str(v)] = statistics.mean(scores)
        print(f"{key} = {str(v):<28} {results[str(v)]:>10,.0f}")
    return results


# Examples — run one at a time, they are slow.
sweep("max_hands", [6, 10, 14])
# sweep("min_price_frac", [0.2, 0.35, 0.5])
# sweep("crop_mix", [
#     {"CARROT": 0.40, "WHEAT": 0.25, "MELON": 0.25, "TOMATO": 0.10},
#     {"CARROT": 0.55, "WHEAT": 0.25, "MELON": 0.20},
#     {"CARROT": 0.30, "WHEAT": 0.20, "MELON": 0.35, "TOMATO": 0.15},
# ])

In [ ]:
!tar -czf submission.tar.gz main.py
!ls -la submission.tar.gz

In [ ]:
# Then pull replays and logs of your ladder games to see how you lose:
#
#   kaggle competitions episodes <SUBMISSION_ID> -v
#   kaggle competitions replay <EPISODE_ID> -p ./replays
#   kaggle competitions logs <EPISODE_ID> 0 -p ./logs
print("ready")

## 6. Where the headroom is

Ranked by expected value, roughly:

**Animals.** The baseline ignores them, and they are the highest-yield tiles in
the game. A goose produces 1 egg/tile/day at a $50 base — better than any crop's
yield-times-price. Cows and sheep are slower but their products base at $160 and
$200. The cost is logistics: build the coop or pasture, `PICKUP` the animal from
the shed, `PLACE` it, then `FEED` one wheat per animal per day from a unit's
inventory, and two missed days means the animal escapes permanently. That feed
loop is why the baseline grows wheat it barely profits from. Implementing this
well is probably worth more than everything else on this list combined.

**Fertilizer.** `FERTILIZE` doubles the per-day watering bonus for three days,
and wheat and carrot only reach their listed max yields (6 and 4) with it.
`COLLECT_FERTILIZER` gathers it free from animals, which compounds with the
point above.

**Movement.** Units respawn at the shed every morning, so a large fraction of
every day is walking. The current `_step_toward` is naive. Assigning each unit a
contiguous work zone for the day, and routing to minimise total travel rather
than greedily per-turn, should raise the effective action count substantially.

**Sell timing.** Prices rise as town shops drain inventory and fall as both
players sell. `unlocked_shops` is visible, shop unlocks are every three days
drawn with replacement, and the town center consumes on a fixed daily tick. A
model that predicts price a few hours out and holds premium goods for the peak
is a small supervised problem with a clear payoff — and the public replay
dataset gives you training data for it.

**Watching the opponent.** Both farms are public in `obs["farms"]`. If they are
about to dump 200 melons, you want to sell first.

**The replay dataset.** Rather than guessing at any of the above, filter the
public episode replays to the highest-rated agents and look at what they
actually do: crop mix, hire counts, when they buy land, how their bank curve
diverges from yours. That turns a design problem into a data problem.